In [3]:
# getting the warriors team id
import nba_api
from nba_api.stats.static import teams

nba_teams = teams.get_teams()

# gonna go through the dictionary for the warriors and find the team id
warriors = [team for team in nba_teams if team["abbreviation"] == "GSW"][0]
warriors_id = warriors["id"]
print(f"warriors_id: {warriors_id}")

warriors_id: 1610612744


In [1]:
import sys
print(sys.executable)

/Users/hussaintaheri/Desktop/sports-win-predictor/venv/bin/python


In [5]:
# lets query for the last regular season game where the warriors are playing
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType

gamefinder = leaguegamefinder.LeagueGameFinder(
    team_id_nullable = warriors_id,
    season_nullable = Season.default,
    season_type_nullable = SeasonType.regular
)

games_dict = gamefinder.get_normalized_dict()
games = games_dict["LeagueGameFinderResults"]
game = games[0]
game_id = game["GAME_ID"]
game_matchup = game["MATCHUP"]

print(f"Searching through {len(games)} game(s) for the game_id of {game_id} where {game_matchup}")

Searching through 74 game(s) for the game_id of 0022501073 where GSW vs. WAS


In [6]:
# query for the play by play of the most recent gsw regular season game
from nba_api.stats.endpoints import playbyplayv3

df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
df

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022501073,2,PT12M00.00S,1,0,,0,,,0,...,0,0,0,,Start of 1st Period (10:11 PM EST),period,start,1,0,1
1,0022501073,4,PT12M00.00S,1,1610612744,GSW,204001,Porziņģis,K. Porziņģis,0,...,,,0,h,Jump Ball Porzingis vs. Sarr: Tip to Carrington,Jump Ball,,1,0,2
2,0022501073,7,PT11M43.00S,1,1610612764,WAS,1642267,Carrington,B. Carrington,108,...,0,2,2,v,Carrington 17' Pullup Jump Shot (2 PTS),Made Shot,Pullup Jump shot,1,2,3
3,0022501073,8,PT11M27.00S,1,1610612744,GSW,204001,Porziņģis,K. Porziņģis,141,...,3,2,5,h,Porzingis 27' 3PT Jump Shot (3 PTS) (Santos 1 ...,Made Shot,Jump Shot,1,3,4
4,0022501073,10,PT11M22.00S,1,1610612764,WAS,1642267,Carrington,B. Carrington,0,...,,,0,v,Carrington Out of Bounds - Bad Pass Turnover T...,Turnover,Out of Bounds - Bad Pass Turnover,1,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,0022501073,701,PT00M11.40S,4,1610612744,GSW,1630611,Santos,G. Santos,0,...,131,126,257,h,Santos Free Throw 2 of 2 (27 PTS),Free Throw,Free Throw 2 of 2,1,0,469
469,0022501073,702,PT00M11.40S,4,0,,1610612764,,,0,...,,,0,v,Wizards Timeout: Regular (Reg.7 Short 0),Timeout,Regular,0,0,470
470,0022501073,703,PT00M10.40S,4,1610612764,WAS,1630702,Hardy,J. Hardy,0,...,,,0,v,Hardy Bad Pass Turnover (P1.T11),Turnover,Bad Pass,1,0,471
471,0022501073,703,PT00M10.40S,4,1610612744,GSW,1641764,Podziemski,B. Podziemski,0,...,,,0,h,Podziemski STEAL (2 STL),,,1,0,472


In [7]:
# documentation for the leaguegamefinder parameters
help(leaguegamefinder)

Help on module nba_api.stats.endpoints.leaguegamefinder in nba_api.stats.endpoints:

NAME
    nba_api.stats.endpoints.leaguegamefinder - LeagueGameFinder endpoint for finding games based on various filters.

DESCRIPTION
    Example:
        >>> from nba_api.stats.endpoints import LeagueGameFinder
        >>> # Get Lakers games from 2023-24 season
        >>> games = LeagueGameFinder(
        ...     player_or_team_abbreviation='T',
        ...     team_id_nullable='1610612747',
        ...     season_nullable='2023-24'
        ... )
        >>> df = games.league_game_finder_results.get_data_frame()

    Important Notes:
        **Date Filtering (Issue #526):**
        The date_from_nullable and date_to_nullable parameters work correctly and
        filter games by date range. Use format 'YYYY-MM-DD'.

        Example - filter by date range:
            >>> # Get all games in January 2024
            >>> games = LeagueGameFinder(
            ...     player_or_team_abbreviation='T',
    

In [15]:
# create a csv file to append our data to
filename = "nba_pbp_2024_2025.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2024_2025.csv was created


In [20]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games = []
failed_teams = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2024-25",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games.append(game_id)
                    break
                    
                time_val *= 2

CPU times: user 28.5 s, sys: 4.01 s, total: 32.5 s
Wall time: 49min 51s


In [21]:
len(failed_games), len(failed_teams)

(2, 0)

In [22]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2024_2025.csv", "w")

/var/folders/x8/rmhx8t213ml9vyqwtqy3vm0r0000gn/T/ipykernel_76815/1678194232.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  season_df.to_csv("nba_pbp_2024_2025.csv", "w")


In [23]:
# now lets pull the previous season before that one
# create a csv file to append our data to
filename = "nba_pbp_2023_2024.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2023_2024.csv was created


In [24]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games2 = []
failed_teams2 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2023-24",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams2.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games2.append(game_id)
                    break
                    
                time_val *= 2

CPU times: user 37.5 s, sys: 7.7 s, total: 45.2 s
Wall time: 2h 52min 8s


In [25]:
len(failed_games2), len(failed_teams2)

(14, 0)

In [26]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2023_2024.csv", "w")

/var/folders/x8/rmhx8t213ml9vyqwtqy3vm0r0000gn/T/ipykernel_76815/3267847747.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  season_df.to_csv("nba_pbp_2023_2024.csv", "w")


In [27]:
# create a csv file to append our data to
filename = "nba_pbp_2022_2023.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2022_2023.csv was created


In [28]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games3 = []
failed_teams3 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2022-23",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams3.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games3.append(game_id)
                    break
                    
                time_val *= 2

CPU times: user 32.7 s, sys: 5.49 s, total: 38.2 s
Wall time: 2h 56min


In [29]:
len(failed_games3), len(failed_teams3)

(11, 1)

In [30]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2022_2023.csv", "w")

/var/folders/x8/rmhx8t213ml9vyqwtqy3vm0r0000gn/T/ipykernel_76815/3808525564.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  season_df.to_csv("nba_pbp_2022_2023.csv", "w")


In [31]:
# create a csv file to append our data to
filename = "nba_pbp_2021_2022.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2021_2022.csv was created


In [32]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games4 = []
failed_teams4 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2021-22",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams4.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games4.append(game_id)
                    break
                    
                time_val *= 2

CPU times: user 36.6 s, sys: 5.44 s, total: 42 s
Wall time: 3h 9min 8s


In [33]:
len(failed_games4), len(failed_teams4)

(17, 0)

In [34]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2021_2022.csv", "w")

/var/folders/x8/rmhx8t213ml9vyqwtqy3vm0r0000gn/T/ipykernel_76815/2244679065.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  season_df.to_csv("nba_pbp_2021_2022.csv", "w")


In [35]:
# create a csv file to append our data to
filename = "nba_pbp_2020_2021.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_pbp_2020_2021.csv was created


In [36]:
%%time
# getting every single team id and getting every game id and adding it to our set to avoid getting dups while also pulling the data 
# and using sleep to not get an error for spamming requests
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2020-21",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2

CPU times: user 59.5 s, sys: 12.6 s, total: 1min 12s
Wall time: 3h 47min 51s


In [37]:
len(failed_games5), len(failed_teams5)

(4, 1)

In [38]:
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2020_2021.csv", "w")

/var/folders/x8/rmhx8t213ml9vyqwtqy3vm0r0000gn/T/ipykernel_76815/1275057338.py:2: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.
  season_df.to_csv("nba_pbp_2020_2021.csv", "w")


In [3]:
import pandas as pd
df1 = pd.read_csv("/Users/hussaintaheri/Desktop/sports-win-predictor/data/nba_pbp_2024_2025.csv", sep = "w")
df1.shape

(598873, 25)

In [56]:
df1.head()

,Unnamed: 0,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,...,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0,22401186,2,PT12M00.00S,1,0,NaN,0,NaN,NaN,...,0.0,0.0,0,NaN,Start of 1st Period (1:12 PM EST),period,start,0,0,1
1,1,22401186,4,PT12M00.00S,1,1610612737,ATL,1631230,Barlow,D. Barlow,...,NaN,NaN,0,h,Jump Ball Barlow vs. Carter Jr.: Tip to Mann,Jump Ball,NaN,1,0,2
2,2,22401186,7,PT11M43.00S,1,1610612737,ATL,1629611,Mann,T. Mann,...,NaN,NaN,0,h,MISS Mann 4' Fadeaway Jumper,Missed Shot,Fadeaway Jump Shot,1,2,3
3,3,22401186,8,PT11M40.00S,1,1610612753,ORL,1628976,Carter Jr.,W. Carter Jr.,...,NaN,NaN,0,v,Carter Jr. REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,4
4,4,22401186,9,PT11M23.00S,1,1610612753,ORL,203484,Caldwell-Pope,K. Caldwell-Pope,...,NaN,NaN,0,v,MISS Caldwell-Pope 25' 3PT Jump Shot,Missed Shot,Jump Shot,1,3,5


In [57]:
df1.dtypes

Unnamed: 0          int64
gameId              int64
actionNumber        int64
clock              object
period              int64
teamId              int64
teamTricode        object
personId            int64
playerName         object
playerNameI        object
xLegacy             int64
yLegacy             int64
shotDistance        int64
shotResult         object
isFieldGoal         int64
scoreHome         float64
scoreAway         float64
pointsTotal         int64
location           object
description        object
actionType         object
subType            object
videoAvailable      int64
shotValue           int64
actionId            int64
dtype: object

In [4]:
df2 = pd.read_csv("/Users/hussaintaheri/Desktop/sports-win-predictor/data/nba_pbp_2023_2024.csv", sep = "w")
df2.shape

(545426, 25)

In [5]:
df3 = pd.read_csv("/Users/hussaintaheri/Desktop/sports-win-predictor/data/nba_pbp_2022_2023.csv", sep = "w")
df3.shape

(558813, 25)

In [6]:
df4 = pd.read_csv("/Users/hussaintaheri/Desktop/sports-win-predictor/data/nba_pbp_2021_2022.csv", sep = "w")
df4.shape

(536929, 25)

In [7]:
df5 = pd.read_csv("/Users/hussaintaheri/Desktop/sports-win-predictor/data/nba_pbp_2020_2021.csv", sep = "w")
df5.shape

(505209, 25)

In [8]:
dfs = [df1, df2, df3, df4, df4, df5]

df_all_seasons = pd.concat(dfs)
df_all_seasons.shape

(3282179, 25)

In [10]:
df_all_seasons.to_csv("../data/nba_pbp_2020_2025.csv")

In [2]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2019_2020.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2019-20",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2019_2020.csv", "w")

nba_pbp_2019_2020.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 14 s, sys: 1.87 s, total: 15.9 s
Wall time: 2h 10min 56s


In [3]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2018_2019.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2018-19",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2018_2019.csv", "w")

nba_pbp_2018_2019.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 36.4 s, sys: 3.32 s, total: 39.7 s
Wall time: 3h 52min 35s


In [4]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2017_2018.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2017-18",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2017_2018.csv", "w")

nba_pbp_2017_2018.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 34.5 s, sys: 3.52 s, total: 38 s
Wall time: 42min 57s


In [5]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2016_2017.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2016-17",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2016_2017.csv", "w")

nba_pbp_2016_2017.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 3min 16s, sys: 17.4 s, total: 3min 33s
Wall time: 51min 23s


In [6]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2015_2016.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2015-16",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2015_2016.csv", "w")

nba_pbp_2015_2016.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 33.3 s, sys: 3.09 s, total: 36.4 s
Wall time: 1h 25min 11s


In [7]:
%%time
# we need more data to decrease our log loss metric
import pandas as pd
import nba_api
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.library.parameters import Season, SeasonType
from nba_api.stats.endpoints import playbyplayv3
import time

filename = "nba_pbp_2014_2015.csv"
with open(filename, "w", newline = "") as csvfile:
    pass
print(f"{filename} was created")

nba_teams = teams.get_teams()
games_set = set()
season_dfs = []
t_counter, g_counter = 0, 0
time_val = 1
failed_games5 = []
failed_teams5 = []

for team in nba_teams:
    while True:
        try:
            team_id = team["id"]
            team_games = leaguegamefinder.LeagueGameFinder(
                team_id_nullable = team_id,
                season_nullable = "2014-15",
                season_type_nullable = SeasonType.regular
            )
            time.sleep(time_val)
            games_dict = team_games.get_normalized_dict()
            games = games_dict["LeagueGameFinderResults"]
            time_val = 1
            break
        except:
            if time_val > 33:
                    time_val = 1
                    failed_teams5.append(team_id)
                    break
                    
                    time_val *= 2
        

    for game in games:
        game_id = game["GAME_ID"]
        while True:
            try:
                if game_id in games_set:
                    break
                    
                games_set.add(game_id)
        
                df = playbyplayv3.PlayByPlayV3(game_id).get_data_frames()[0]
                time.sleep(time_val)
                season_dfs.append(df)
                time_val = 1
                break
            except:
                if time_val > 33:
                    time_val = 1
                    failed_games5.append(game_id)
                    break
                    
                time_val *= 2
season_df = pd.concat(season_dfs)
season_df.to_csv("nba_pbp_2014_2015.csv", "w")

nba_pbp_2014_2015.csv was created


<timed exec>:68: FutureWarning: Starting with pandas version 3.0 all arguments of to_csv except for the argument 'path_or_buf' will be keyword-only.


CPU times: user 2min 18s, sys: 23.3 s, total: 2min 41s
Wall time: 3h 9min 46s
